## EDA on label from User

QUESTION 1: Find all possible unique values
- Category
- Subcategory
- Root_code
- Sub_code

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('label.xlsx')

print("--- 1. UNIQUE VALUES IN DATASET ---")
print(f"Categories ({df['Category'].nunique()}): {df['Category'].dropna().unique()}")
print(f"Subcategories ({df['Subcategory'].nunique()}): {df['Subcategory'].dropna().unique()}")
print(f"Root Codes ({df['root_code'].nunique()}): {df['root_code'].dropna().unique()}")
print(f"Sub Codes ({df['sub_code'].nunique()}): {df['sub_code'].dropna().unique()}\n")

QUESTION 2: Find mappings and human errors

In [ ]:
print("--- 2. MAPPING RULES & ANOMALY DETECTION ---")

# Group by the human text labels and see what codes they mapped to, counting occurrences
mapping_df = df.groupby(
    ['Category', 'Subcategory', 'root_code', 'sub_code'], 
    dropna=False
).size().reset_index(name='Count')

# Sort so that for each Category/Subcategory, the most frequent mapping is at the top
mapping_df = mapping_df.sort_values(
    by=['Category', 'Subcategory', 'Count'], 
    ascending=[True, True, False]
)

# Calculate the total times a Category/Subcategory pair appears
mapping_df['Total_Category_Occurrences'] = mapping_df.groupby(['Category', 'Subcategory'])['Count'].transform('sum')

# Calculate the percentage of times THIS specific mapping was used
mapping_df['Mapping_Confidence_%'] = (mapping_df['Count'] / mapping_df['Total_Category_Occurrences']) * 100

print("\nAll Mappings (Sorted by frequency):")
display(mapping_df) # Use print(mapping_df) if not in Jupyter

In [ ]:
# --- ISOLATING THE ERRORS ---
# Assuming that whatever mapping happens the majority of the time (>50%) is the "Correct" rule...
# Anything else is likely a human annotation error.

print("\n--- LIKELY HUMAN ANNOTATION ERRORS ---")
# Filter for mappings that happen less than 20% of the time for that specific category
anomalies = mapping_df[mapping_df['Mapping_Confidence_%'] < 20.0]

if anomalies.empty:
    print("Good news! No major anomalies found. The mapping is consistent.")
else:
    print("Found potential mislabels! Annotators likely messed these up:")
    display(anomalies)

In [ ]:
# ==========================================
# CELL 1: EYEBALL SPECIFIC CATEGORIES
# ==========================================
from IPython.display import Image, display
import os

# 1. Define the specific pair you want to investigate
TARGET_CATEGORY = "Insert Category Here"       # e.g., "Medical Document"
TARGET_SUBCAT = "Insert Subcategory Here"      # e.g., "Lab Results"

# 2. Filter the existing 'df' for this specific pair
subset_df = df[(df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)]

print(f"Found {len(subset_df)} documents for {TARGET_CATEGORY} -> {TARGET_SUBCAT}\n")

# 3. Display the images (Showing the first 10 to avoid crashing the notebook)
for idx, row in subset_df.head(10).iterrows():
    print("-" * 50)
    print(f"Row Index: {idx}")
    print(f"Current Labels -> root_code: [{row['root_code']}] | sub_code: [{row['sub_code']}]")
    print(f"Filepath: {row['Filepath']}")
    
    filepath = str(row['Filepath'])
    if os.path.exists(filepath):
        # Display the image directly in the notebook output
        display(Image(filename=filepath, width=600))
    else:
        print(f"Image not found at path: {filepath}")

In [ ]:
# ==========================================
# CELL 2: BATCH REPLACE & SAVE
# ==========================================

# 1. Define the pair you want to fix
TARGET_CATEGORY = "Insert Category Here"
TARGET_SUBCAT = "Insert Subcategory Here"

# 2. Define the CORRECT codes they should be mapped to
CORRECT_ROOT_CODE = "NON"
CORRECT_SUB_CODE = "OTH"

# 3. Create the filter condition
condition = (df['Category'] == TARGET_CATEGORY) & (df['Subcategory'] == TARGET_SUBCAT)

# 4. Use .loc to safely update the values in the original dataframe
df.loc[condition, 'root_code'] = CORRECT_ROOT_CODE
df.loc[condition, 'sub_code'] = CORRECT_SUB_CODE

print(f"Successfully updated {condition.sum()} rows to {CORRECT_ROOT_CODE} -> {CORRECT_SUB_CODE}.")

# 5. Export the cleaned data to CSV
output_filename = 'label_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"Cleaned dataset exported as: {output_filename}")

## Evaluation

In [ ]:
# ==========================================
# v11 CHANGELOG (vs v10) -- ARCHITECTURE RESTRUCTURE
# ==========================================
# Old structure: Triage (medical / processable_non_medical / trash_others) ->
#   Non-Med Specialist (financial_* / id_*) OR Med Specialist (medical_*)
#
# New structure, per team decision to resolve the financial_others / id_others /
# trash_others mix-up:
#   Step 1 - Triage (binary): medical / non_medical
#   Step 2a - if medical: Medical Specialist (UNCHANGED, still owned by your senior)
#   Step 2b - if non_medical: Non-Medical Router: financial / identification / eform /
#             unrelated_document
#   Step 3a - if financial: Financial Specialist -> bankstatement / bookbank /
#             companyregistration / receipt / selfincomedeclaration / others
#   Step 3b - if identification: Identification Specialist -> the 10 existing id_*
#             leaves + 2 new ones (id_houseregistration, id_marriagecertificate) --
#             flagged as tentative, see note below function definitions.
#   eform and unrelated_document are terminal at Step 2 -- no Step 3 call for those.
#
# What moved where:
#   - The old Triage's eForm litmus test, financial anchors, and ID anchors all moved
#     into the new Non-Medical Router (Step 2b), since that's now the stage that
#     actually has to draw the financial/identification/eform/unrelated line.
#   - Step 1 Triage is now much shorter: since "processable vs trash" no longer exists
#     as a Step-1 decision, all of that anchor logic became irrelevant to Step 1. It only
#     needs the medical-vs-not-medical criteria, which were already the clearest part of
#     the old prompt.
#   - Hospital receipts moved from "unrelated_document" into Financial (receipt).
# ==========================================

import math
import time
import random
import os
import csv
import threading
import pandas as pd
from typing import Literal, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel
from openai import AzureOpenAI, BadRequestError, RateLimitError, APIConnectionError, APITimeoutError

try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

# ==========================================
# 1. DEFINE STRUCTURED OUTPUT SCHEMAS
# ==========================================

class TriageOutput(BaseModel):
    chain_of_thought: str
    routing_decision: Literal["medical", "non_medical"]


class NonMedicalRouterOutput(BaseModel):
    chain_of_thought: str
    document_class: Literal["financial", "identification", "eform", "unrelated_document"]


class FinancialSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "financial_bankstatement", "financial_bookbank", "financial_companyregistration",
        "financial_receipt", "financial_selfincomedeclaration", "financial_others"
    ]


class IdentificationSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "id_driverlicense", "id_fatca_w9", "id_foreignerconfirmationform",
        "id_foreigner_nationalid", "id_visastamp", "id_passport",
        "id_statelessid", "id_thainationalid", "id_workpermit",
        "id_houseregistration", "id_marriagecertificate",  # NEW, tentative -- see note below
        "id_others"
    ]


class MedSpecialistOutput(BaseModel):
    chain_of_thought: str
    subcategory: Literal[
        "medical_clinical", "medical_healthcheck",
        "medical_lab", "medical_others"
    ]

# ==========================================
# 2. DEFINE PROMPT VARIABLES
# ==========================================

# --- STEP 1: BINARY TRIAGE ---
# Massively simplified vs the old 3-way triage. All the anchor logic for distinguishing
# processable-ID/financial from trash is no longer Step 1's job -- both outcomes are now
# "non_medical" and get sorted out downstream by the Non-Medical Router. Step 1 only
# needs to answer one question: is genuine clinical content present?
AGENT1_SYS_PROMPT = """You are a Triage Routing Agent. Classify raw OCR/markdown text into exactly one of two categories: `medical` or `non_medical`. Ignore PII.

### medical
Requires the **clinical content itself**: clinical notes, lab metrics, health-check parameters, prescriptions, test ranges, normal/abnormal flags. Not a reference to it, not correspondence about it.
- **INCLUDE:** Garbled OCR that still shows technical-looking medical terms next to numbers/units/ranges, or positive/negative or normal/abnormal flags -- treat as low-confidence medical rather than defaulting away from it just because the scan is blurry. Flag the uncertainty in chain_of_thought.
- **EXCLUDE -> non_medical:** Billing, invoices, or receipts (hospital or otherwise). Any letter, memo, consent form, or eForm that discusses, requests, or forwards medical information without containing the clinical data itself -- true even if it's hospital-stamped or has a signature block. Ask: is this the clinical record, or correspondence *about* one?

### non_medical
Default for everything else: identification documents, financial documents, insurance eForms, receipts, unrelated/illegible documents, correspondence, and anything that isn't genuine clinical content. You do not need to determine which of these it is -- a downstream specialist handles that distinction. If you're unsure and the clinical-content bar above isn't clearly met, route non_medical.

In chain_of_thought: state whether genuine clinical content was found, and if not, briefly note what the document appears to be instead."""

AGENT1_USER_PROMPT = "Evaluate and triage the following raw OCR text:\n{ocr_text}"


# --- STEP 2b: NON-MEDICAL ROUTER ---
# This absorbs the old Triage's eForm-vs-financial-vs-ID logic (litmus test, anchors,
# watermark handling), restructured for the new 4-way taxonomy. Two directives from the
# team decision are load-bearing here: (1) hospital receipts are now Financial, not
# unrelated_document; (2) a form-shaped document with genuine income/numeric substance
# routes to Financial even if it superficially looks like an eform.
AGENT2_SYS_PROMPT = """You are a Non-Medical Document Router. You receive OCR/markdown text (often with layout tags -- headers, tables, <figure> regions) already confirmed as non-medical. Classify it into exactly one of: `financial`, `identification`, `eform`, `unrelated_document`. Treat the patterns below as supporting evidence, not an exact-string checklist -- read holistically. Ignore PII.

### Ignore Watermarks
Insurance disclaimers (+ date/time stamps) are never evidence of document type -- ignore them completely.

### financial
- Bank/account identity (account number + bank name/branch), or a native transaction ledger (dated money movements, ideally a markdown table).
- Corporate registration number + certification/signature block.
- Any receipt or proof-of-payment document -- retail, invoice, billing, **including hospital/medical receipts.** Receipts are financial, not unrelated_document.
- Self income declaration: the document's actual substance is a person reporting, clarifying, or declaring their own income, salary (เงินเดือน), or source of income -- even if it's laid out as a form or questionnaire. Content decides this, not layout.
- General agreements/contracts: lease, sale, or other business contracts.

### identification
A genuine ID/travel document with its own native structure (not a field filled into someone else's form):
- Issuing country name/code (e.g. "LAO PDR", "RDP LAO"), a genuine name+DOB or name+issue/expiry block, "รับรองสำเนาถูกต้อง" (certified true copy), or a `<figure>` region actually wrapping such a personal-detail block -- not just a name/reference number sitting near a `<figure>` tag with no other identity structure.
- Stateless-ID text, or garbled non-Thai script (e.g. Lao misread as Thai) combined with a country-code header.
- MRZ lines (letters/digits + long "<" runs, e.g. "PA123SD464987<<<<<<<"), or immigration/visa keywords -- "VISA", "VISA CLASS"/"VISACLASS", "IMMIGRATION", "DEPARTED", "ADMITTED", "ENTRY" -- near messy digits/dates. These are standardized terms, safe to trust literally even amid heavy OCR noise; don't let a "looks like noise" impression override them.
- FATCA/W-9 tax-withholding declaration forms count as identification here, not financial.

### eform
Insurance-related administrative paperwork with no independent financial or identity substance of its own:
- Insurance application/policy forms concerning เบี้ยประกัน or policy details -- fields filled in about an applicant, not the applicant's own native ID/financial document.
- Documents opening with "บันทึกคำชี้แจงเกี่ยวกับใบคำขอประกันภัย" or similar clarification-note language about an insurance application.
- Consent letters for data collection, or document/information request letters (e.g. asking for additional documents).
- **Exception -- read this carefully:** if a form-shaped document's actual content is income/numeric substance (an income questionnaire, salary clarification, source-of-income disclosure), route to `financial` (self income declaration) instead. The form layout alone does not make something an eform; the litmus test is what the content actually says once you look past the layout.

### unrelated_document
Default when nothing above applies: portrait photos, ID placeholders with no issuing text, illegible noise, postal/mailing-label text that describes a document without containing it, or anything else lacking every pattern above.

In chain_of_thought: name the pattern found (or "none"), and explicitly note if a form-shaped document was routed to financial instead of eform because its content was genuinely income-related, or vice versa."""

AGENT2_USER_PROMPT = "Classify the following non-medical OCR text into financial, identification, eform, or unrelated_document:\n{ocr_text}"


# --- STEP 3a: FINANCIAL SPECIALIST ---
AGENT_FINANCIAL_SYS_PROMPT = """You are an expert Financial Document Specialist. You receive text already confirmed as financial. Perform granular classification into one of six leaf classes. Use markdown structure (tables, headers) where present. Do not process PII.

- financial_bankstatement: running transactional ledgers, money transfers, account balances, mobile banking markers -- ideally a markdown table of dated transaction rows.
- financial_bookbank: static account identity headers (bilingual accounts, branches, bank names) without a long transaction ledger.
- financial_companyregistration: formal commercial corporate registry verbiage and certification signatures.
- financial_receipt: any proof-of-payment document -- retail receipts, invoices, billing statements, or hospital/medical receipts.
- financial_selfincomedeclaration: the person reporting, clarifying, or declaring their own income, salary, or source of income -- regardless of whether it's laid out as a form or questionnaire.
- financial_others: financial in nature but doesn't cleanly match the above -- also covers general agreements/contracts (lease, sale, business contracts), and is the safe landing spot if you suspect this document was mis-routed here from an eform with no independent financial substance.

Provide step-by-step structural rationale in chain_of_thought before selecting the final leaf subcategory."""

AGENT_FINANCIAL_USER_PROMPT = "Perform deep financial classification on this verified text:\n{ocr_text}"


# --- STEP 3b: IDENTIFICATION SPECIALIST ---
# NOTE: id_houseregistration and id_marriagecertificate are new leaves per your team's
# tentative plan to split these out of id_others. They're included below since you gave
# concrete Thai terms for both, but this is the one part of this restructure I'd treat as
# "confirm before you commit ground-truth labels to it" -- easy to fold back into
# id_others (just delete these two lines and remove them from the schema) if your team
# decides against the split.
AGENT_ID_SYS_PROMPT = """You are an expert Identification Document Specialist. You receive text already confirmed as an identification/travel document. Perform granular classification. Use markdown structure where present. Do not process PII.

- id_thainationalid / id_foreigner_nationalid: national demographic card headers and identity issuing text blocks. Foreign-script documents (e.g. a Lao national ID) still count even if largely unreadable by OCR -- look for legible fragments like issuing country name/code, name, or a DOB pattern.
- id_passport: the passport bio data page -- photo/data block, passport number, nationality, an MRZ line (letters/digits + long "<" runs).
- id_visastamp: a Thai visa page and/or immigration control stamp -- visa category codes, "DEPARTED"/"ADMITTED"/"ENTRY" keywords, and (often garbled, since stamp text is curved) date-like digit strings near them.
- id_workpermit: labor authorization rules and official employment permission context.
- id_fatca_w9: US tax withholding terminology, backup withholding rules, entity declaration sections.
- id_foreignerconfirmationform: an official government-issued confirmation/declaration form attesting to a foreign national's identity or status -- issued BY a government/authority ABOUT the person, not filled in by an agent for a policy application.
- id_statelessid: a stateless-person identity card -- card layout similar to a national ID (photo placeholder, ID number, name, DOB) but with issuing text indicating stateless/no-registered-nationality status.
- id_driverlicense: driver's license.
- id_houseregistration: household registration document (ทะเบียนบ้าน) -- household registration book header, house/address registry formatting, list of household members.
- id_marriagecertificate: marriage certificate (ใบทะเบียนสมรส) -- marriage registration terminology, groom/bride names, registrar signature, marriage date.
- id_others: clearly an ID-type artifact that doesn't cleanly match any subtype above. Expected, valid outcome for borderline documents -- don't force a wrong specific subtype, and this is also the safe landing spot if you suspect a mis-routed eform with no independent identity substance.

Provide step-by-step structural rationale in chain_of_thought before selecting the final leaf subcategory."""

AGENT_ID_USER_PROMPT = "Perform deep identification classification on this verified text:\n{ocr_text}"


# --- MEDICAL SPECIALIST PROMPTS (UNCHANGED -- owned by a different prompt owner) ---
AGENT3_SYS_PROMPT = """You are an expert Medical Document Specialist. You receive text that has already been verified as an official medical or healthcare document. Your task is to perform granular classification.

Analyze clinical boilerplate language, diagnostic formats, and structural indicators. Do not attempt to analyze or extract confidential patient names or raw medical history.

### Subclass Evaluation Guides:
- medical_clinical: Look for outpatient/inpatient notes (OPD/IPD), physical exam details, hospital visit dates, physician signatures, and medication dosages/frequencies (e.g., "mg", "ครั้ง").
- medical_healthcheck: Look for standardized checkup checklists, physiological summary metrics (e.g., "min", "bpm"), and categorical evaluations of bodily systems (e.g., "normal", "abnormal", "ปกติ", "ไม่พบ").
- medical_lab: Look for quantitative lab panels, chemical compound analysis, test execution timestamps, measurement metrics (e.g., "mg/dL"), or diagnostic presence indicators (e.g., "positive", "negative").
- medical_others: Select this if the document is clearly medical but does not visually or structurally conform to an OPD/IPD sheet, a regular checkup report, or a quantitative lab metric page.

Provide your step-by-step structural rationale in 'chain_of_thought' before selecting the final leaf node subcategory."""

AGENT3_USER_PROMPT = "Perform deep clinical classification on this verified medical text:\n{ocr_text}"


# ==========================================
# 3. CORE COGNITIVE PIPELINE
# ==========================================

def calculate_confidence(response_logprobs) -> float:
    """Whole-response confidence (legacy). Kept for backward compatibility / comparison.
    NOTE: this dilutes the signal with free-text chain_of_thought tokens -- prefer
    calculate_field_confidence for anything you actually act on (e.g. review-queue thresholds)."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0
    token_logprobs = [t.logprob for t in response_logprobs.content if t.logprob is not None]
    if not token_logprobs:
        return 0.0
    avg_logprob = sum(token_logprobs) / len(token_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def calculate_field_confidence(response_logprobs, field_value: str) -> float:
    """Isolates and averages logprobs only for the tokens making up the classification
    enum value itself, rather than the whole JSON payload including the free-text
    chain_of_thought. Falls back to calculate_confidence if the token span can't be
    located, so it always returns a usable number."""
    if not response_logprobs or not response_logprobs.content:
        return 0.0

    tokens = response_logprobs.content
    token_strs = [t.token for t in tokens]

    running = ""
    span_start, span_end = None, None
    for i, tok in enumerate(token_strs):
        prev_len = len(running)
        running += tok
        if field_value in running[max(0, prev_len - len(field_value)):]:
            end_idx = i
            partial = ""
            start_idx = i
            for j in range(i, -1, -1):
                partial = token_strs[j] + partial
                start_idx = j
                if field_value in partial:
                    break
            span_start, span_end = start_idx, end_idx

    if span_start is None:
        return calculate_confidence(response_logprobs)

    span_logprobs = [tokens[i].logprob for i in range(span_start, span_end + 1) if tokens[i].logprob is not None]
    if not span_logprobs:
        return calculate_confidence(response_logprobs)

    avg_logprob = sum(span_logprobs) / len(span_logprobs)
    return round(math.exp(avg_logprob) * 100, 2)


def call_with_retry(fn, *args, max_retries: int = 6, base_delay: float = 1.0,
                     max_delay: float = 60.0, **kwargs):
    """Wraps a single API call with retry + exponential backoff for 429s and transient
    connection errors. Respects Azure's Retry-After header when present."""
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except RateLimitError as e:
            retry_after = None
            try:
                retry_after = float(e.response.headers.get("retry-after"))
            except Exception:
                pass
            delay = retry_after if retry_after is not None else min(max_delay, base_delay * (2 ** attempt))
            delay += random.uniform(0, delay * 0.25)
            print(f"[RATE LIMIT] attempt {attempt + 1}/{max_retries}, sleeping {delay:.1f}s")
            time.sleep(delay)
        except (APIConnectionError, APITimeoutError) as e:
            delay = min(max_delay, base_delay * (2 ** attempt)) + random.uniform(0, 1)
            print(f"[TRANSIENT ERROR] {type(e).__name__}: {e} -- sleeping {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"Exceeded max_retries ({max_retries}) calling {getattr(fn, '__name__', fn)}")


def run_cascade_pipeline(ocr_text: str, client: AzureOpenAI,
                          triage_model: str = "gpt-5-mini",
                          router_model: str = "gpt-5-mini",
                          financial_model: str = "gpt-5",
                          id_model: str = "gpt-5",
                          med_model: str = "gpt-5") -> dict:
    """Executes the new multi-stage architecture on a single piece of text.

    Step 1 (triage_model): medical vs non_medical.
    Step 2a (med_model): if medical, classify into medical leaf directly.
    Step 2b (router_model): if non_medical, route into financial / identification /
        eform / unrelated_document.
    Step 3 (financial_model or id_model): only called if Step 2b said financial or
        identification. eform and unrelated_document are terminal -- no Step 3 call.

    Default model tiering: triage and the router are both coarse decisions on
    already-good-accuracy stages, so cheaper models there; the two Step 3 specialists
    get the stronger model since granular leaf classification is where accuracy is
    hardest to get right.
    """

    # ----------------------------------------------------
    # STEP 1: Triage (medical vs non_medical)
    # ----------------------------------------------------
    triage_response = call_with_retry(
        client.beta.chat.completions.parse,
        model=triage_model,
        messages=[
            {"role": "system", "content": AGENT1_SYS_PROMPT},
            {"role": "user", "content": AGENT1_USER_PROMPT.format(ocr_text=ocr_text)}
        ],
        response_format=TriageOutput,
        logprobs=True,
        top_logprobs=1
    )
    triage_data = triage_response.choices[0].message.parsed
    route = triage_data.routing_decision
    triage_conf = calculate_field_confidence(triage_response.choices[0].logprobs, route)

    base_result = {
        "triage_decision": route,
        "triage_reason": triage_data.chain_of_thought,
        "triage_confidence": triage_conf,
        "router_decision": None,
        "router_reason": None,
        "router_confidence": None,
    }

    # ----------------------------------------------------
    # STEP 2a: Medical -> classify directly (unchanged from prior architecture)
    # ----------------------------------------------------
    if route == "medical":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=med_model,
            messages=[
                {"role": "system", "content": AGENT3_SYS_PROMPT},
                {"role": "user", "content": AGENT3_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=MedSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)

        return {
            **base_result,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf,
        }

    # ----------------------------------------------------
    # STEP 2b: Non-medical -> route into financial / identification / eform / unrelated
    # ----------------------------------------------------
    router_response = call_with_retry(
        client.beta.chat.completions.parse,
        model=router_model,
        messages=[
            {"role": "system", "content": AGENT2_SYS_PROMPT},
            {"role": "user", "content": AGENT2_USER_PROMPT.format(ocr_text=ocr_text)}
        ],
        response_format=NonMedicalRouterOutput,
        logprobs=True,
        top_logprobs=1
    )
    router_data = router_response.choices[0].message.parsed
    doc_class = router_data.document_class
    router_conf = calculate_field_confidence(router_response.choices[0].logprobs, doc_class)

    base_result["router_decision"] = doc_class
    base_result["router_reason"] = router_data.chain_of_thought
    base_result["router_confidence"] = router_conf

    # ---- STEP 3a: financial ----
    if doc_class == "financial":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=financial_model,
            messages=[
                {"role": "system", "content": AGENT_FINANCIAL_SYS_PROMPT},
                {"role": "user", "content": AGENT_FINANCIAL_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=FinancialSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)
        return {
            **base_result,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf,
        }

    # ---- STEP 3b: identification ----
    if doc_class == "identification":
        spec_response = call_with_retry(
            client.beta.chat.completions.parse,
            model=id_model,
            messages=[
                {"role": "system", "content": AGENT_ID_SYS_PROMPT},
                {"role": "user", "content": AGENT_ID_USER_PROMPT.format(ocr_text=ocr_text)}
            ],
            response_format=IdentificationSpecialistOutput,
            logprobs=True,
            top_logprobs=1
        )
        spec_data = spec_response.choices[0].message.parsed
        spec_conf = calculate_field_confidence(spec_response.choices[0].logprobs, spec_data.subcategory)
        return {
            **base_result,
            "final_subcategory": spec_data.subcategory,
            "specialist_reason": spec_data.chain_of_thought,
            "specialist_confidence": spec_conf,
        }

    # ---- Terminal: eform / unrelated_document -- no Step 3 call ----
    return {
        **base_result,
        "final_subcategory": doc_class,
        "specialist_reason": "No further specialization needed -- terminal at the router stage.",
        "specialist_confidence": router_conf,
    }


# ==========================================
# 4. DATAFRAME EVALUATION EXECUTION
# ==========================================

_checkpoint_lock = threading.Lock()

_ERROR_FALLBACK_EXTRA = {
    "router_decision": None, "router_reason": None, "router_confidence": None,
}


def _process_one_row(idx, row, client, triage_model, router_model, financial_model, id_model, med_model):
    """Runs the cascade pipeline for a single row, with the same error handling as
    before (content filter block vs. any other exception)."""
    ocr_text = str(row['ocr_text'])

    try:
        res = run_cascade_pipeline(ocr_text, client, triage_model, router_model, financial_model, id_model, med_model)
    except BadRequestError as e:
        error_message = str(e).lower()
        if "content management policy" in error_message or "jailbreak" in error_message:
            print(f"[WARNING] Doc {idx + 1} blocked by Azure Jailbreak Filter. Skipping...")
            res = {
                "triage_decision": "blocked_by_firewall",
                "triage_reason": "Azure OpenAI Content Filter flagged this OCR text as a jailbreak attempt.",
                "triage_confidence": 0.0,
                **_ERROR_FALLBACK_EXTRA,
                "final_subcategory": "error_content_filter",
                "specialist_reason": "Skipped due to API security policy.",
                "specialist_confidence": 0.0
            }
        else:
            raise e
    except Exception as e:
        print(f"[ERROR] Doc {idx + 1} failed due to unexpected error: {e}")
        res = {
            "triage_decision": "api_error",
            "triage_reason": str(e),
            "triage_confidence": 0.0,
            **_ERROR_FALLBACK_EXTRA,
            "final_subcategory": "api_error",
            "specialist_reason": "API disconnected or failed.",
            "specialist_confidence": 0.0
        }

    combined_row = {
        "filename": row.get('filename'),
        "input_category_label": row.get('category'),
        "input_subcategory_label": row.get('subcategory'),
        **res
    }
    return idx, combined_row


def evaluate_experiment(csv_path: str, endpoint: str, api_key: str,
                         triage_model: str = "gpt-5-mini",
                         router_model: str = "gpt-5-mini",
                         financial_model: str = "gpt-5",
                         id_model: str = "gpt-5",
                         med_model: str = "gpt-5",
                         max_workers: int = 5,
                         checkpoint_path: str = "experiment_full_cascade_results.csv",
                         resume: bool = True):
    """Reads the evaluation data, executes the cascade pipeline concurrently, and
    returns results.

    triage_model / router_model / financial_model / id_model / med_model: deployment
        names for each stage. These must match deployment names in your Azure resource.
    max_workers: number of documents processed in parallel. Start conservative (3-5) and
        raise it while watching for [RATE LIMIT] log lines.
    checkpoint_path: results are appended here as each document finishes.
    resume: if True and checkpoint_path already exists, filenames already present there
        are skipped instead of being re-processed.
    """

    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-08-01-preview"
    )

    df = pd.read_csv(csv_path)

    already_done = set()
    if resume and os.path.isfile(checkpoint_path):
        prior = pd.read_csv(checkpoint_path)
        if 'filename' in prior.columns:
            already_done = set(prior['filename'].dropna().astype(str))
        print(f"Resuming: {len(already_done)} document(s) already in checkpoint, will be skipped.")

    pending = [
        (idx, row) for idx, row in df.iterrows()
        if str(row.get('filename')) not in already_done
    ]

    print(f"Starting cascade experiment on {len(pending)} document(s) "
          f"(of {len(df)} total, {len(already_done)} already done) with max_workers={max_workers}...")

    fieldnames = None

    def write_row(combined_row):
        nonlocal fieldnames
        with _checkpoint_lock:
            file_exists = os.path.isfile(checkpoint_path)
            if fieldnames is None:
                if file_exists:
                    fieldnames = list(pd.read_csv(checkpoint_path, nrows=0).columns)
                else:
                    fieldnames = list(combined_row.keys())
            with open(checkpoint_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(combined_row)

    results_by_idx = {}
    completed = 0
    iterator_kwargs = dict(total=len(pending)) if HAS_TQDM else {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_one_row, idx, row, client,
                             triage_model, router_model, financial_model, id_model, med_model): idx
            for idx, row in pending
        }

        progress = tqdm(as_completed(futures), **iterator_kwargs) if HAS_TQDM else as_completed(futures)

        for future in progress:
            idx, combined_row = future.result()
            write_row(combined_row)
            results_by_idx[idx] = combined_row
            completed += 1
            if not HAS_TQDM:
                print(f"[{completed}/{len(pending)}] {combined_row.get('filename')} "
                      f"Triage: {combined_row['triage_decision']} -> Router: {combined_row.get('router_decision')} "
                      f"-> Leaf: {combined_row['final_subcategory']}")

    print(f"\nDone. {completed} document(s) processed this run. "
          f"Full results (including any resumed rows) are in '{checkpoint_path}'.")

    output_df = pd.read_csv(checkpoint_path)
    return output_df

# To run in your notebook cell:
# df_results = evaluate_experiment(
#     "test_data.csv",
#     "https://your-endpoint.openai.azure.com/",
#     "your-api-key",
#     triage_model="gpt-5-mini",
#     router_model="gpt-5-mini",
#     financial_model="gpt-5",
#     id_model="gpt-5",
#     med_model="gpt-5",     # swap to whatever your senior wants for medical
#     max_workers=5,
# )

In [ ]:
def evaluate_experiment(csv_path: str, endpoint: str, api_key: str,
                         triage_model: str = "gpt-5-mini",
                         nonmed_model: str = "gpt-5",
                         med_model: str = "gpt-5",
                         max_workers: int = 5,
                         checkpoint_path: str = "experiment_full_cascade_results.csv",
                         resume: bool = True):
    """Reads the evaluation data, executes the cascade pipeline concurrently, and
    returns results.
 
    triage_model / nonmed_model / med_model: deployment names for each stage. Defaults
        reflect a cost/performance split -- cheaper model where accuracy is already good
        (triage), stronger model where accuracy is the actual bottleneck (specialists).
        These must match deployment names that exist in your Azure resource, not the
        raw OpenAI model family names -- rename to whatever you deployed them as.
    max_workers: number of documents processed in parallel. Start conservative (3-5) and
        raise it while watching for [RATE LIMIT] log lines -- if you see frequent 429s
        even after backoff, max_workers is higher than your Azure deployment's TPM/RPM
        quota supports. Check your quota under Azure OpenAI Studio > Quotas.
    checkpoint_path: results are appended here as each document finishes, so a crash
        partway through doesn't lose completed work.
    resume: if True and checkpoint_path already exists, filenames already present there
        are skipped instead of being re-processed.
    """
 
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2024-08-01-preview"
    )
 
    df = pd.read_csv(csv_path)
 
    already_done = set()
    if resume and os.path.isfile(checkpoint_path):
        prior = pd.read_csv(checkpoint_path)
        if 'filename' in prior.columns:
            already_done = set(prior['filename'].dropna().astype(str))
        print(f"Resuming: {len(already_done)} document(s) already in checkpoint, will be skipped.")
 
    pending = [
        (idx, row) for idx, row in df.iterrows()
        if str(row.get('filename')) not in already_done
    ]
 
    print(f"Starting cascade experiment on {len(pending)} document(s) "
          f"(of {len(df)} total, {len(already_done)} already done) with max_workers={max_workers}...")
 
    fieldnames = None  # inferred from the first completed row
 
    def write_row(combined_row):
        nonlocal fieldnames
        with _checkpoint_lock:
            file_exists = os.path.isfile(checkpoint_path)
            if fieldnames is None:
                if file_exists:
                    fieldnames = list(pd.read_csv(checkpoint_path, nrows=0).columns)
                else:
                    fieldnames = list(combined_row.keys())
            with open(checkpoint_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(combined_row)
 
    results_by_idx = {}
    completed = 0
    iterator_kwargs = dict(total=len(pending)) if HAS_TQDM else {}
 
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_one_row, idx, row, client, triage_model, nonmed_model, med_model): idx
            for idx, row in pending
        }
 
        progress = tqdm(as_completed(futures), **iterator_kwargs) if HAS_TQDM else as_completed(futures)
 
        for future in progress:
            idx, combined_row = future.result()
            write_row(combined_row)
            results_by_idx[idx] = combined_row
            completed += 1
            if not HAS_TQDM:
                print(f"[{completed}/{len(pending)}] {combined_row.get('filename')} "
                      f"Triage: {combined_row['triage_decision']} -> Leaf: {combined_row['final_subcategory']}")
 
    print(f"\nDone. {completed} document(s) processed this run. "
          f"Full results (including any resumed rows) are in '{checkpoint_path}'.")
 
    output_df = pd.read_csv(checkpoint_path)
    return output_df
 
# To run in your notebook cell:
# df_results = evaluate_experiment(
#     "test_data.csv",
#     "https://your-endpoint.openai.azure.com/",
#     "your-api-key",
#     triage_model="gpt-5-mini",   # deployment name for the cheap/already-accurate stage
#     nonmed_model="gpt-5",        # deployment name for the accuracy bottleneck stage
#     med_model="gpt-5",           # swap to whatever your senior wants for medical
#     max_workers=5,                # tune to your Azure quota
# )